In [1]:
# Cell 00: Colab Stuff
import os

DEVELOPMENT_MODE = False

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    os.system('pip install --upgrade numpy --quiet')
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

In [2]:
# Cell 0: Imports
import sys
import torch
from pathlib import Path

# Colab: files are in /content/
# Local: notebook is in notebooks/, project root is one level up
if Path('/content/data').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/trishasalas/Repos/Research/tmlr


In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [4]:
model_name = "EleutherAI/pythia-2.8b"
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("pythia-2.8b", device=device)
model.eval()

print(f"n_layers: {model.cfg.n_layers}")
print(f"d_model: {model.cfg.d_model}")
print(f"d_mlp: {model.cfg.d_mlp}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-2.8b into HookedTransformer
n_layers: 32
d_model: 2560
d_mlp: 10240


In [ ]:
#prompt = "A long navigation menu without a skip link is not accessible because"
#prompt = "A website without screen reader support is not accessible because"
prompt = "An image without alt text is not accessible because"

tokens = model.to_tokens(prompt)
print(f"prompt tokens: {tokens.shape}")
print(f"token strings: {model.to_str_tokens(prompt)}")

output = model.generate(prompt, max_new_tokens=50, temperature=0, verbose=False)
print("---")
print(output)

prompt tokens: torch.Size([1, 10])
token strings: ['<|endoftext|>', 'An', ' image', ' without', ' alt', ' text', ' is', ' not', ' accessible', ' because']
---
An image without alt text is not accessible because it is not clickable.

The image is not accessible because it is not clickable.

The image is not accessible because it is not clickable.

The image is not accessible because it is not clickable.




In [12]:
# Generate with tokens visible one at a time
input_ids = model.to_tokens(prompt)
generated_tokens = []

with torch.no_grad():
    current_ids = input_ids.clone()
    for step in range(10):
        logits = model(current_ids)
        next_token_logits = logits[0, -1, :]
        next_token = torch.argmax(next_token_logits).item()
        next_token_str = model.to_string(torch.tensor([next_token]))
        generated_tokens.append((step, next_token, next_token_str, next_token_logits[next_token].item()))
        current_ids = torch.cat([current_ids, torch.tensor([[next_token]], device=device)], dim=1)

print(f"{'step':<5} {'token_id':<10} {'string':<20} {'logit':<10}")
print("-" * 50)
for step, tok_id, tok_str, logit in generated_tokens:
    print(f"{step:<5} {tok_id:<10} {repr(tok_str):<20} {logit:<10.3f}")

step  token_id   string               logit     
--------------------------------------------------
0     352        ' it'                16.208    
1     310        ' is'                17.757    
2     417        ' not'               16.849    
3     5532       ' click'             15.627    
4     494        'able'               21.515    
5     15         '.'                  17.986    
6     187        '\n'                 14.703    
7     187        '\n'                 16.851    
8     510        'The'                13.908    
9     2460       ' image'             12.977    


In [13]:
# What's competing at the decision point right after "because"?
input_ids = model.to_tokens(prompt)

with torch.no_grad():
    logits = model(input_ids)
    step0_logits = logits[0, -1, :]

# Top 15 candidates
top_values, top_indices = torch.topk(step0_logits, 15)

print(f"{'rank':<6} {'token_id':<10} {'string':<20} {'logit':<10}")
print("-" * 50)
for rank, (val, idx) in enumerate(zip(top_values, top_indices)):
    tok_str = model.to_string(torch.tensor([idx.item()]))
    print(f"{rank:<6} {idx.item():<10} {repr(tok_str):<20} {val.item():<10.3f}")

rank   token_id   string               logit     
--------------------------------------------------
0      352        ' it'                16.208    
1      253        ' the'               15.858    
2      627        ' there'             14.425    
3      273        ' of'                14.329    
4      697        ' its'               13.495    
5      368        ' you'               13.288    
6      247        ' a'                 13.036    
7      6945       ' alt'               12.979    
8      634        ' your'              12.941    
9      436        ' this'              12.882    
10     642        ' no'                12.804    
11     359        ' we'                12.588    
12     326        ' that'              12.546    
13     187        '\n'                 12.382    
14     690        ' some'              12.205    
